In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import ripser
import persim

In [2]:
########## read the data ##########

processes = [f"run_{i:02d}_decayed_1" for i in range(1,12)]
kappa_3 = [0.2, 0.4, 0.6, 0.8, 0.9, 1, 1.1, 1.2, 1.4, 1.6, 1.8]
# Xsection_NoDecay = [1.23322e-01, 1.016366e-01, 8.33476424e-02, 6.81961e-02, 6.2211286e-02, 5.648398e-02, 5.187586e-02, 4.801255e-02, 4.246736e-02, 4.038406e-02, 4.1308219e-02]     ### hhvv unit: fb
Xsection_NoDecay = [1.068573e-2, 8.89267e-3, 7.29573e-3, 5.892737e-3, 5.2836183e-3, 4.68848e-3, 4.1895e-3, 3.7137245e-3, 2.9218193e-3, 2.3228e-3, 1.9414667e-3]     ### hhmumu unit: fb
BR_hbb = 0.5824
Xsection = []
for i in range(11):
    Xsection.append(Xsection_NoDecay[i]*BR_hbb*BR_hbb)
    
Luminosity = 1000   ### unit: fb^-1
simulation_num = 100000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Mass", "VLCjetR10N2.Constituents", "EFlowTrack.fUniqueID", "EFlowTrack.PT", "EFlowTrack.Eta", "EFlowTrack.Phi", "EFlowTrack.PID", "EFlowPhoton.fUniqueID", "EFlowPhoton.ET", "EFlowPhoton.Eta", "EFlowPhoton.Phi", "EFlowNeutralHadron.fUniqueID", "EFlowNeutralHadron.ET", "EFlowNeutralHadron.Eta", "EFlowNeutralHadron.Phi", "VLCjetR10N2.Eta"]

##### set data path #####

event_path = []
for type in processes:
    event_path.append("/data/mucollider/two_boosted/3TeV/scan_kappa3_hhmumu/Events/" + type + "/delphes_output_v2.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print(process, "Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

/usr/local/lib/python3.8/dist-packages/awkward/array/base.py:394: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return cls.numpy.array(value, copy=False)


run_01_decayed_1 Time:23.8710 (s)
run_02_decayed_1 Time:46.0695 (s)
run_03_decayed_1 Time:61.2138 (s)
run_04_decayed_1 Time:83.6328 (s)
run_05_decayed_1 Time:105.9223(s)
run_06_decayed_1 Time:122.9628(s)
run_07_decayed_1 Time:145.0396(s)
run_08_decayed_1 Time:164.4659(s)
run_09_decayed_1 Time:181.9831(s)
run_10_decayed_1 Time:200.7148(s)
run_11_decayed_1 Time:224.8072(s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if Eta_J < XX #####

def Eta_selection(events, eta_max):
    where1 = []
    for i in range(len(events)):
        if np.all(np.abs(events[i][features.index("VLCjetR10N2.Eta")]) <= eta_max):
            where1.append(i)
    return events[where1]

In [4]:
def get_particle_info(event):
    ##### set jet constituents ID #####

    leadingJ_constituents = event[features.index("VLCjetR10N2.Constituents")][0][:-1]
    subleadingJ_constituents= event[features.index("VLCjetR10N2.Constituents")][1][:-1]

    ##### construct infromation for all EFlow particles in this events and sort by PT #####

    all_ID = np.concatenate((event[features.index("EFlowTrack.fUniqueID")], event[features.index("EFlowPhoton.fUniqueID")], event[features.index("EFlowNeutralHadron.fUniqueID")]))
    all_PT = np.concatenate((event[features.index("EFlowTrack.PT")], event[features.index("EFlowPhoton.ET")], event[features.index("EFlowNeutralHadron.ET")]))
    all_Eta = np.concatenate((event[features.index("EFlowTrack.Eta")], event[features.index("EFlowPhoton.Eta")], event[features.index("EFlowNeutralHadron.Eta")]))
    all_Phi = np.concatenate((event[features.index("EFlowTrack.Phi")], event[features.index("EFlowPhoton.Phi")], event[features.index("EFlowNeutralHadron.Phi")]))
    all_PID = np.concatenate((event[features.index("EFlowTrack.PID")], np.full(len(event[features.index("EFlowPhoton.fUniqueID")]), 22), np.full(len(event[features.index("EFlowNeutralHadron.fUniqueID")]), 4)))

    #####                          categorize PID                      #####
    ##### 1:electron 2:muon 3:photon 4:neutral hadron 5:charged hadron #####

    condlist = [abs(all_PID)==11, abs(all_PID)==13, abs(all_PID)==22, abs(all_PID)==4]
    choicelist = [1, 2, 3, 4]
    all_PID = np.select(condlist, choicelist, 5)

    sort_order = np.argsort(all_PT)[::-1]   ### from large PT to low PT

    all_ID = all_ID[sort_order]
    all_PT = all_PT[sort_order]
    all_Eta = all_Eta[sort_order]
    all_Phi = all_Phi[sort_order]
    all_PID = all_PID[sort_order]

    ##### find Jet particle in EFlow particle #####
    if len(leadingJ_constituents)>20: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0][0:20]
    else: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0]

    if len(subleadingJ_constituents)>20: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0][0:20]
    else: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0]

    leadingJ_PT, subleadingJ_PT = all_PT[where_leading], all_PT[where_subleading]
    leadingJ_Eta, subleadingJ_Eta = all_Eta[where_leading], all_Eta[where_subleading]
    leadingJ_Phi, subleadingJ_Phi = all_Phi[where_leading], all_Phi[where_subleading]
    leadingJ_PID, subleadingJ_PID = all_PID[where_leading], all_PID[where_subleading]

    leadingJ_info, subleadingJ_info = np.stack((leadingJ_PT, leadingJ_Eta, leadingJ_Phi, leadingJ_PID), axis=1).ravel(), np.stack((subleadingJ_PT, subleadingJ_Eta, subleadingJ_Phi, subleadingJ_PID), axis=1).ravel()

    ##### do zero padding #####

    particle_info = np.zeros(2*20*4)
    particle_info[0:len(leadingJ_info)] = leadingJ_info
    particle_info[80:80+len(subleadingJ_info)] = subleadingJ_info
    
    return particle_info

In [5]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### jets Eta selection #####

eta_max = -np.log(np.tan(np.radians(10) / 2))
for process in processes:
    events[processes.index(process)] = Eta_selection(events[processes.index(process)], eta_max)
print("\nAfter Eta selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 850   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 100000 run_01_decayed_1 events. Corresponding cross section: 0.0036244901941248 (fb)
There are 100000 run_02_decayed_1 events. Corresponding cross section: 0.0030163026030592005 (fb)
There are 100000 run_03_decayed_1 events. Corresponding cross section: 0.0024746369077248004 (fb)
There are 100000 run_04_decayed_1 events. Corresponding cross section: 0.00199875604877312 (fb)
There are 100000 run_05_decayed_1 events. Corresponding cross section: 0.001792149223108608 (fb)
There are 100000 run_06_decayed_1 events. Corresponding cross section: 0.0015902844059648004 (fb)
There are 100000 run_07_decayed_1 events. Corresponding cross section: 0.00142103549952 (fb)
There are 100000 run_08_decayed_1 events. Corresponding cross section: 0.0012596573218611202 (fb)
There are 100000 run_09_decayed_1 events. Corresponding cross section: 0.000991051187130368 (fb)
There are 100000 run_10_decayed_1 events. Corresponding cross section: 0.000787869974528 (fb)
There are 1000

In [6]:
##### generate particle information #####

particle_info = []

for process in processes:
    for event in events[processes.index(process)]:
        particle_info.append(get_particle_info(event))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:271.3634(s)
Time:278.4935(s)
Time:285.7817(s)
Time:293.0244(s)
Time:299.9302(s)
Time:306.8659(s)
Time:313.6495(s)
Time:320.2048(s)
Time:326.9211(s)
Time:333.3503(s)
Time:339.4820(s)


In [7]:
particle_info[0]

array([ 3.48973198e+01,  8.84436131e-01,  6.06276333e-01,  1.00000000e+00,
        2.41519260e+01,  1.49775708e+00,  3.52352411e-01,  3.00000000e+00,
        2.31132030e+01,  1.52621353e+00,  4.23146904e-01,  5.00000000e+00,
        1.85214710e+01,  1.51544523e+00,  3.86045307e-01,  5.00000000e+00,
        1.67007923e+01,  1.49623048e+00,  3.93703520e-01,  5.00000000e+00,
        1.60293694e+01,  1.49828875e+00,  3.44455540e-01,  5.00000000e+00,
        1.56102448e+01,  8.40615869e-01,  6.93303049e-01,  4.00000000e+00,
        1.55353861e+01,  8.52893174e-01,  6.84656203e-01,  5.00000000e+00,
        1.55106373e+01,  8.94142449e-01,  6.09010160e-01,  5.00000000e+00,
        1.53033323e+01,  1.48350489e+00,  3.93493325e-01,  3.00000000e+00,
        1.26048326e+01,  1.51569676e+00,  3.80250394e-01,  3.00000000e+00,
        1.20716667e+01,  1.53427064e+00,  5.11448383e-01,  5.00000000e+00,
        1.14958506e+01,  1.46359503e+00,  3.97566319e-01,  3.00000000e+00,
        1.14223700e+01,  

In [8]:
##### output test data #####

d = {'particle_info': particle_info}
df = pd.DataFrame(data=d)
df

,particle_info
0,"[34.89731979370117, 0.8844361305236816, 0.6062..."
1,"[43.58861541748047, -1.1702187061309814, -2.41..."
2,"[36.00285720825195, -0.9769997000694275, -1.63..."
3,"[116.0334243774414, 0.2646521031856537, 0.0898..."
4,"[75.28253936767578, -1.0498672723770142, 2.053..."
...,...
314761,"[109.92391967773438, 0.09465201944112778, -1.2..."
314762,"[35.58210372924805, 1.4534392356872559, -1.662..."
314763,"[62.64446258544922, 1.6173290014266968, 0.0663..."
314764,"[34.398921966552734, 0.318581759929657, -0.569..."


In [9]:
import h5py
h5f = h5py.File('/data/mucollider/two_boosted/3TeV/scan_kappa3_hhmumu_LL_v2.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")
h5f.close()

已寫入 Dataset: particle_info
